# Stacking with an MLP Meta-Model on the SOTA Ensemble (CIFAR-10)

## Goal

The SOTA soft-voting ensemble (ResNet18 + DenseNet121) reaches **96.87%** test
(and **97.12%** with 2-view TTA). Instead of a fixed `0.5 + 0.5` soft-voting
rule, **stacking** trains a small **MLP meta-model** to *learn* how to combine
the two base models' probability vectors.

We compare two approaches, both with a baseline of the plain SOTA ensemble:
1. **SOTA ensemble + TTA flip + MLP** — base probabilities are TTA-averaged
   (horizontal flip, 2 views) before the MLP combines them.
2. **SOTA ensemble + MLP** — base probabilities are single-pass (no TTA).

## Stacking / meta-model idea

$$\mathbf{p}_{\text{meta}}(x) = \operatorname{MLP}\big([\,p_{\text{ResNet}}(x),\, p_{\text{DenseNet}}(x)\,]\big)$$

The MLP input is the **concatenated 20-dim** softmax of the two base models
(10 + 10). It can learn *per-model, per-class* trust (e.g. "trust ResNet on
cats, DenseNet on ships") rather than a fixed 0.5/0.5 blend.

## Leakage-aware protocol
- Base models: `ResNet18-sota` / `DenseNet121-sota` (unchanged, from checkpoints).
- The **MLP meta-model is trained on the validation split** (5k) and evaluated
  on the **test split** (10k) — never trained on test.
- Each approach trains its own meta-model on its own feature type (TTA vs not).

---

## References

- [LOGGING_CHECKPOINT_RULES.md](../agents/rules/LOGGING_CHECKPOINT_RULES.md)
- [RESULTS_REPORTING.md](../agents/rules/RESULTS_REPORTING.md)
- [stacking_mlp_train.py](../src/experiments/stacking_mlp_train.py) — training entry point (script)
- [checkpoint_utils.py](../src/utils/checkpoint_utils.py) — artifact loading
- Artifacts: [experiments/results/stacking_mlp/](../experiments/results/stacking_mlp/), [experiments/plots/stacking_mlp_confusion.png](../experiments/plots/stacking_mlp_confusion.png)

## 1. Setup & Data

We need the **validation** split (to train the meta-model) and the **test**
split (to evaluate), both at 224x224 with the ImageNet eval transform.

In [ ]:
import json, sys, os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Robust project-root detection (works from repo root or notebooks/).
_cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = _cwd if (_cwd / "src").exists() else (_cwd.parent if (_cwd.parent / "src").exists() else _cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.build_model import build_resnet18, build_densenet121
from src.eval.evaluate_model import CIFAR10_CLASSES
from src.utils.checkpoint_utils import find_best_checkpoint, load_model_weights

CAT_IDX, DOG_IDX = 3, 5
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD
DATA_ROOT = str(PROJECT_ROOT / "data" / "raw")
SPLIT_FILE = str(PROJECT_ROOT / "data" / "processed" / "cifar10_split_seed42.json")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

tform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
split = json.loads(Path(SPLIT_FILE).read_text())

def make_loader(train, indices, batch_size=64):
    ds = torchvision.datasets.CIFAR10(DATA_ROOT, train=train, transform=tform)
    if indices is not None:
        ds = Subset(ds, indices)
    return DataLoader(ds, batch_size=batch_size, shuffle=False)

val_loader  = make_loader(True,  split["val_indices"])    # train the meta-model here
test_loader = make_loader(False, None)                    # evaluate here
print(f"val samples: {len(val_loader.dataset)}, test samples: {len(test_loader.dataset)}")

## 2. Load the SOTA Base Models & Helpers

`base_features(model_a, model_b, loader, tta)` returns the concatenated
20-dim `[p_ResNet, p_DenseNet]` features for every sample, optionally with
2-view horizontal-flip TTA.

In [ ]:
# ---- Load SOTA base models ----
ckpt = PROJECT_ROOT / "experiments" / "checkpoints"   # legacy flat fallback
rn = build_resnet18(num_classes=10, mode="finetune", device=device)
dn = build_densenet121(num_classes=10, mode="finetune", device=device)
rn = load_model_weights(rn, find_best_checkpoint("ResNet18-sota", fallback_dir=ckpt), device)
dn = load_model_weights(dn, find_best_checkpoint("DenseNet121-sota", fallback_dir=ckpt), device)
rn.eval(); dn.eval()
print("Loaded ResNet18-sota and DenseNet121-sota.")

# (feature extraction now runs in src/experiments/stacking_mlp_train.py —
#  this notebook loads the persisted artifacts only)
# (feature extraction now runs in src/experiments/stacking_mlp_train.py —
#  this notebook loads the persisted artifacts only)

## 3. Extract Base Features (val for training, test for eval)

Feature extraction and meta-model **training run as a script** —
`python -m src.experiments.stacking_mlp_train`. This notebook only **loads** the
persisted artifacts (`experiments/results/stacking_mlp/`) and analyzes them.
The 20-dim features are `concat([p_ResNet, p_DenseNet])` softmax probabilities
per sample, computed on the val split (train the meta-model) and test split
(evaluate), each with and without 2-view hflip TTA.

In [ ]:
# ---- Load persisted features (extraction ran as a script) ----
art_path = PROJECT_ROOT / "experiments" / "results" / "stacking_mlp" / "stacking_mlp_artifacts.npz"
print(f"[load] stacking features -> {art_path}")
art = np.load(art_path)
X_val, y_val = art["X_val"], art["y_val"]
X_valT = art["X_valT"]
X_test, y_test = art["X_test"], art["y_test"]
X_testT = art["X_testT"]
print("val features:", X_val.shape, " test features:", X_test.shape)
print("(feature dim = 20 = 10 ResNet + 10 DenseNet probabilities)")

## 4. Baseline — fixed soft-voting (no MLP), with and without TTA

Reference points we want the MLP approaches to beat.

In [ ]:
def full_metrics(probs, targets):
    preds = np.argmax(probs, axis=1)
    acc = (preds == targets).mean() * 100.0
    cm = confusion_matrix(targets, preds, labels=list(range(10)))
    rep = classification_report(targets, preds, labels=list(range(10)),
                                output_dict=True, zero_division=0)
    return acc, rep["macro avg"]["f1-score"], cm

def isolated_catdog(probs, targets):
    mask = (targets == CAT_IDX) | (targets == DOG_IDX)
    t, p = targets[mask], np.argmax(probs[mask], axis=1)
    acc = (p == t).mean() * 100.0
    cross = int(((t == CAT_IDX) & (p == DOG_IDX)).sum()
                + ((t == DOG_IDX) & (p == CAT_IDX)).sum())
    return acc, cross

art2 = np.load(PROJECT_ROOT / "experiments" / "results" / "stacking_mlp" / "stacking_mlp_artifacts.npz")
p_ens, p_ensT = art2["p_ens"], art2["p_ensT"]

acc_base, f1_base, _ = full_metrics(p_ens, y_test)
acc_baseT, f1_baseT, _ = full_metrics(p_ensT, y_test)
iso_base, cross_base = isolated_catdog(p_ens, y_test)
iso_baseT, cross_baseT = isolated_catdog(p_ensT, y_test)

print(f"Soft-voting (no TTA): full_acc={acc_base:.2f}%  f1={f1_base:.4f}  "
      f"isolated={iso_base:.2f}%  cross={cross_base}")
print(f"Soft-voting + TTA   : full_acc={acc_baseT:.2f}%  f1={f1_baseT:.4f}  "
      f"isolated={iso_baseT:.2f}%  cross={cross_baseT}")

## 5. The Stacking MLP Meta-Model

A small MLP `[20 -> 64 -> ReLU -> 10]`. Trained on the **validation** features
with cross-entropy, evaluated on the **test** features. We train one on the
non-TTA features (approach 2) and one on the TTA features (approach 1).

In [ ]:
class StackingMLP(nn.Module):
    def __init__(self, in_dim=20, hidden=64, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        return self.net(x)

# ---- Load trained meta-models (training ran as a script) ----
# python -m src.experiments.stacking_mlp_train
art_dir = PROJECT_ROOT / "experiments" / "results" / "stacking_mlp"
mlp_notta = StackingMLP(in_dim=X_val.shape[1]).to(device)
mlp_tta = StackingMLP(in_dim=X_val.shape[1]).to(device)
print(f"[load] stacking MLP weights -> {art_dir / 'mlp_notta.pt'}, {art_dir / 'mlp_tta.pt'}")
mlp_notta.load_state_dict(torch.load(art_dir / "mlp_notta.pt", map_location=device, weights_only=True))
mlp_tta.load_state_dict(torch.load(art_dir / "mlp_tta.pt", map_location=device, weights_only=True))
print(f"[load] stacking config -> {art_dir / 'config.json'}")
cfg = json.loads((art_dir / "config.json").read_text())
val_acc_notta = cfg["val_acc_notta"]
val_acc_tta = cfg["val_acc_tta"]
print(f"Meta-model val acc (no TTA): {val_acc_notta:.2f}%  (trained by script)")
print(f"Meta-model val acc (with TTA): {val_acc_tta:.2f}%")

## 6. Approach 2 — SOTA ensemble + MLP (no TTA)

Apply the single-pass MLP meta-model to the test features.

In [ ]:
with torch.no_grad():
    logits = mlp_notta(torch.tensor(X_test, dtype=torch.float32).to(device))
    p_mlp = torch.softmax(logits, dim=1).cpu().numpy()

acc_mlp, f1_mlp, cm_mlp = full_metrics(p_mlp, y_test)
iso_mlp, cross_mlp = isolated_catdog(p_mlp, y_test)
print(f"Ensemble + MLP (no TTA) : full_acc={acc_mlp:.2f}%  f1={f1_mlp:.4f}  "
      f"isolated={iso_mlp:.2f}%  cross={cross_mlp}   (dAcc={acc_mlp-acc_base:+.2f}%)")

## 7. Approach 1 — SOTA ensemble + TTA flip + MLP

Apply the TTA-trained MLP meta-model to the TTA test features.

In [ ]:
with torch.no_grad():
    logitsT = mlp_tta(torch.tensor(X_testT, dtype=torch.float32).to(device))
    p_mlpT = torch.softmax(logitsT, dim=1).cpu().numpy()

acc_mlpT, f1_mlpT, cm_mlpT = full_metrics(p_mlpT, y_test)
iso_mlpT, cross_mlpT = isolated_catdog(p_mlpT, y_test)
print(f"Ensemble + TTA + MLP     : full_acc={acc_mlpT:.2f}%  f1={f1_mlpT:.4f}  "
      f"isolated={iso_mlpT:.2f}%  cross={cross_mlpT}   (dAcc={acc_mlpT-acc_base:+.2f}%)")

## 8. Comparison — all four configurations

Baseline (fixed 0.5/0.5 soft-voting) vs the two MLP stacking approaches,
with and without TTA.

In [ ]:
rows = [
    ("Soft-voting (no TTA)",   acc_base,  iso_base,  cross_base),
    ("Soft-voting + TTA",      acc_baseT, iso_baseT, cross_baseT),
    ("Ensemble + MLP (no TTA)", acc_mlp,  iso_mlp,   cross_mlp),
    ("Ensemble + TTA + MLP",   acc_mlpT,  iso_mlpT,  cross_mlpT),
]
print(f"{'Method':26s} {'full_acc':>9s} {'dAcc':>7s} {'isolated':>9s} {'cross':>6s}")
for name, acc, iso, cross in rows:
    print(f"{name:26s} {acc:8.2f}% {acc-acc_base:+6.2f}% {iso:8.2f}% {cross:>5d}")

best_name = max(rows, key=lambda r: r[1])[0]
print(f"\nBest: {best_name}")

# Confusion matrices for the two MLP approaches side by side
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, cm, title in [
        (axes[0], cm_mlp,  f"Ensemble + MLP (no TTA) — {acc_mlp:.2f}%"),
        (axes[1], cm_mlpT, f"Ensemble + TTA + MLP — {acc_mlpT:.2f}%")]:
    im = ax.imshow(cm, cmap="Blues"); plt.colorbar(im, ax=ax)
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(CIFAR10_CLASSES, fontsize=7)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title, fontweight="bold")
    for r in range(10):
        for c in range(10):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center", fontsize=6,
                    color="white" if cm[r, c] > cm.max() * 0.6 else "black")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "experiments" / "plots" / "stacking_mlp_confusion.png",
            dpi=300, bbox_inches="tight")
plt.show()

## 9. Discussion & Conclusions

- **Does the MLP meta-model beat fixed soft-voting?** Compare `Ensemble + MLP`
  vs `Soft-voting` (same TTA setting).
- **Does TTA still help on top of stacking?** Compare `Ensemble + TTA + MLP`
  vs `Ensemble + MLP`.
- The MLP is trained on the **validation** split and evaluated on **test** —
  no test leakage. Note the base models used the validation split during their
  own early stopping, so for a fully clean meta-training set one would hold out
  data the base models never saw; this notebook uses the standard
  train-meta-on-val / evaluate-on-test protocol.
- Stacking adds a tiny inference step (one small MLP forward) and can recover
  a bit more than the fixed 0.5/0.5 rule because it learns per-model, per-class
  trust — and it composes with TTA.